In [ ]:
import pandas as pd
import numpy as np

For the private starts we take Started - Private Enterprises.

For social starts we sum Started - Housing Associations and Started - Local Authorities

In [ ]:
# CONFIG
IN = "../../data"

df = pd.read_excel(f"{IN}/raw/starts/indicatorsofukhousebuilding.xlsx",
                   sheet_name="1b",
                   skiprows=5)

# Replacing ONS placeholder with low number - 5
df["Started - Local Authorities"] = df["Started - Local Authorities"].replace("[low]", 5)
df["Started - Local Authorities"] = pd.to_numeric(df["Started - Local Authorities"], errors="coerce")
df["Started - Housing Associations"] = pd.to_numeric(df["Started - Housing Associations"], errors="coerce")


df["lhsoc"] = np.log(df["Started - Housing Associations"] + df["Started - Local Authorities"])
df["lhstarts"] = np.log(df["Started - Private Enterprise"])

cols_to_keep = ["Period", "lhstarts", "lhsoc"]

starts = df[cols_to_keep].copy()

starts = starts.set_index("Period")

# Mapping quarters
quarter_map = {
    "Jan - Mar": "Q1",
    "Apr - Jun": "Q2",
    "Jul - Sep": "Q3",
    "Oct - Dec": "Q4"
}

# Clean up index
idx_str = starts.index.astype(str).str.strip()

# Extract the month range text and the 4-digit year
months = idx_str.str.extract(r'(Jan - Mar|Apr - Jun|Jul - Sep|Oct - Dec)')[0]
years = idx_str.str.extract(r'(\d{4})')[0]

# Translate the months into Q1, Q2, etc.
quarters = months.map(quarter_map)

# Glue into strings
quarter_strings = years + quarters

# Convert to Period objects
starts.index = pd.PeriodIndex(quarter_strings, freq='Q')

# Naming index
starts.index.name = 'Date'

We confirm I(1) - I should test trend stationarity as well

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

adf_stat, adf_p, *_ = adfuller(starts["lhsoc"], autolag="AIC")
kpss_stat, kpss_p, *_ = kpss(starts["lhsoc"], regression="c", nlags="auto")

print("Log Level Test")
print("ADF: ", adf_stat, adf_p)
print("KPSS: ", kpss_stat, kpss_p)

starts["dlhsoc"] = starts["lhsoc"].diff()
starts["dlhstarts"] = starts["lhstarts"].diff()

adf_stat, adf_p, *_ = adfuller(starts["dlhsoc"].dropna(), autolag="AIC")
kpss_stat, kpss_p, *_ = kpss(starts["dlhsoc"].dropna(), regression="c", nlags="auto")

print(f"\nDifference Log Test")
print("ADF: ", adf_stat, adf_p)
print("KPSS: ", kpss_stat, kpss_p)

Remember we found that private starts is I(0) so we need to specify with constant and trend

In [ ]:
from statsmodels.tsa.api import VAR

var_data = starts[["lhstarts", "lhsoc"]].dropna()

model = VAR(var_data)
lag_order_results = model.select_order(maxlags=8, trend="ct")  
# print(lag_order_results.summary())

p = 5
results = model.fit(p)
# print(results.summary())

# Residual autocorrelation check
# print(results.test_whiteness(nlags=8))

# Todo Yamamoto Refit
d_max = 1
p_aug = p + d_max # 6

ty_model = VAR(var_data)
ty_results = ty_model.fit(p_aug, trend="ct")
print(ty_results.summary())

# Residual autocorrelation test
print(ty_results.test_whiteness(nlags=8))


Todo Yamamoto Wald Test for Granger Casuality - Bidirectional

In [ ]:
import numpy as np
from scipy import stats as sp_stats

def ty_granger_wald(results, p, cause, effect):
    """
    Toda-Yamamoto Granger non-causality Wald test.
    Tests H0: 'cause' does not Granger-cause 'effect'.
    """
    cov_full = results.cov_params()
    mi = cov_full.index  # MultiIndex (coef_name, equation)

    coef_names = [f"L{lag}.{cause}" for lag in range(1, p + 1)]
    target_tuples = [(name, effect) for name in coef_names]

    # Positions in the MultiIndex
    positions = [mi.get_loc(t) for t in target_tuples]

    # Coefficient values for the effect equation
    beta_eq = results.params[effect]  # Series indexed by coef name
    beta_restricted = beta_eq.loc[coef_names].values

    cov_vals = cov_full.values

    R = np.zeros((len(positions), cov_vals.shape[0]))
    for i, pos in enumerate(positions):
        R[i, pos] = 1

    middle = R @ cov_vals @ R.T
    wald_stat = beta_restricted.T @ np.linalg.inv(middle) @ beta_restricted
    df = len(coef_names)
    p_value = 1 - sp_stats.chi2.cdf(wald_stat, df)

    return wald_stat, df, p_value

p = 5
wald_soc_to_starts, df1, pval1 = ty_granger_wald(ty_results, p, cause="lhsoc", effect="lhstarts")
wald_starts_to_soc, df2, pval2 = ty_granger_wald(ty_results, p, cause="lhstarts", effect="lhsoc")

print(f"lhsoc -> lhstarts:  Wald = {wald_soc_to_starts:.3f}, df = {df1}, p = {pval1:.4f}")
print(f"lhstarts -> lhsoc:  Wald = {wald_starts_to_soc:.3f}, df = {df2}, p = {pval2:.4f}")

Controlled Regression Pre-Processing

In [ ]:
import statsmodels.api as sm
controls = pd.read_csv(f"{IN}/python_master/england_master.csv")

# Filter to 1978Q1 and onwards

controls["Unnamed: 0"] = pd.PeriodIndex(controls["Unnamed: 0"], freq="Q")
controls = controls.rename(columns={"Unnamed: 0": "Date"})
controls = controls.set_index("Date")

# Filter to 1978Q1 onward
controls = controls[controls.index >= pd.Period("1978Q1", freq="Q")]

controls = controls.drop(columns=["starts", "p_def"])

# Tansformations
controls["lrprc"] = np.log(controls["hprice"] / controls["cc_def"])    # real house prices
controls["lrcc"] = np.log(controls["cc"] / controls["cc_def"])          # real construction costs
controls["lvol"] = np.log(controls["vol"])                         # transactions volume
controls["lstock"] = np.log(controls["hstock"])                    # housing stock
controls["r3"] = controls["rate"]                                   # interest rate



df = controls.join(starts, how="inner")

cols_to_keep = [
    "lhstarts", "lhsoc", "dlhstarts", "dlhsoc",
    "lrprc", "lvol", "r3", "lstock", "lrcc"
]

master = df[cols_to_keep].copy()
master = master.dropna()

Contemporaneous Regression

In [ ]:
y = master["dlhstarts"]
X = master[["dlhsoc", "lrprc", "lvol", "r3", "lrcc"]]
X = sm.add_constant(X)

model = sm.OLS(y, X)
results = model.fit(cov_type="HAC", cov_kwds={"maxlags": 5})  # match VAR p, or justify separately

print(results.summary())

Adding lagged dhsoc as a control

In [ ]:
master["dlhsoc_L1"] = master["dlhsoc"].shift(1)
master["dlhsoc_L2"] = master["dlhsoc"].shift(2)

master_lag = master.dropna()  # drops the new leading NaNs from shift()

y = master_lag["dlhstarts"]
X = master_lag[["dlhsoc", "dlhsoc_L1", "dlhsoc_L2", "lrprc", "lvol", "r3", "lrcc"]]
X = sm.add_constant(X)

model = sm.OLS(y, X)
results_lag = model.fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(results_lag.summary())

In [ ]:
# Dropping contemporaneous term and letting only lag
X_lagonly = master_lag[["dlhsoc_L1", "dlhsoc_L2", "lrprc", "lvol", "r3", "lrcc"]]
X_lagonly = sm.add_constant(X_lagonly)
results_lagonly = sm.OLS(y, X_lagonly).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(results_lagonly.summary())

In [ ]:
import matplotlib.pyplot as plt

window = 4
smoothed = master[["dlhstarts", "dlhsoc"]].rolling(window).mean()

fig, ax = plt.subplots(figsize=(10, 4.5))
x = smoothed.index.to_timestamp()

ax.plot(x, smoothed["dlhstarts"], label="Private starts (4q avg)", color="#1b5e20", linewidth=1.8)
ax.plot(x, smoothed["dlhsoc"], label="Social starts (4q avg)", color="#bdbdbd", linewidth=1.8)

ax.axhline(0, color="black", linewidth=0.6)

ax.set_ylabel("Quarterly change (log, 4q avg)")
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()